# 指标计算模块演示 (Metrics Module)

本notebook演示hscredit库中指标计算模块的全部功能。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")

数据形状: (12448, 85)


In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
demo_feature = feature_cols[0]

# 准备数据
X = df[feature_cols]
y = df[target_col]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

训练集: (8713, 80), 测试集: (3735, 80)


## 1. 导入指标计算模块

In [3]:
from hscredit.core.metrics import (
    KS, AUC, Gini, PSI, IV,
    KS_bucket, ROC_curve,
    confusion_matrix, classification_report,
    PSI_table, CSI_table,
    IV_table,
    MSE, MAE, RMSE, R2
)

print("指标计算模块导入成功！")

指标计算模块导入成功！


## 2. 分类指标

KS、AUC、Gini等风控核心指标。

In [4]:
from sklearn.linear_model import LogisticRegression
from hscredit.core.encoders import WOEEncoder

# WOE编码
woe = WOEEncoder()
X_train_woe = woe.fit_transform(X_train, y_train)
X_test_woe = woe.transform(X_test)

# 训练模型
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_woe, y_train)
y_pred_proba = lr.predict_proba(X_test_woe)[:, 1]

# 计算分类指标
ks_score = KS(y_test, y_pred_proba)
auc_score = AUC(y_test, y_pred_proba)
gini_score = Gini(y_test, y_pred_proba)

print("风控核心指标:")
print(f"  KS: {ks_score:.4f}")
print(f"  AUC: {auc_score:.4f}")
print(f"  Gini: {gini_score:.4f}")

风控核心指标:
  KS: 0.1641
  AUC: 0.4936
  Gini: -0.0127


## 3. 混淆矩阵和分类报告

In [5]:
# 预测类别
y_pred = (y_pred_proba > 0.5).astype(int)

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵:")
print(cm)

# 分类报告
report = classification_report(y_test, y_pred)
print("\n分类报告:")
print(report)

混淆矩阵:
[[3471   17]
 [ 208   39]]

分类报告:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      3488
           1       0.70      0.16      0.26       247

    accuracy                           0.94      3735
   macro avg       0.82      0.58      0.61      3735
weighted avg       0.93      0.94      0.92      3735



## 4. 稳定性指标

PSI和CSI稳定性指标。

In [6]:
# PSI计算
psi_score = PSI(y_train, y_test)
print(f"总体PSI: {psi_score:.4f}")

# 特征PSI
feature_psi = PSI_table(X_train[[demo_feature]], X_test[[demo_feature]])
print(f"\n{demo_feature} PSI分箱表:")
print(feature_psi)

总体PSI: 0.0000

bj_qy24 PSI分箱表:
                 分箱区间  期望样本数  实际样本数   期望占比   实际占比  PSI贡献
0     (-inf, 512.000]    843    382 0.0968 0.1023 0.0003
1  (512.000, 543.000]    906    349 0.1040 0.0934 0.0011
2  (543.000, 563.000]    882    344 0.1012 0.0921 0.0009
3  (563.000, 581.000]    866    397 0.0994 0.1063 0.0005
4  (581.000, 599.000]    873    376 0.1002 0.1007 0.0000
5  (599.000, 615.000]    866    369 0.0994 0.0988 0.0000
6  (615.000, 633.000]    861    365 0.0988 0.0977 0.0000
7  (633.000, 655.000]    901    376 0.1034 0.1007 0.0001
8  (655.000, 683.000]    829    392 0.0951 0.1050 0.0010
9     (683.000, +inf)    886    385 0.1017 0.1031 0.0000


In [7]:
# 批量计算特征PSI
psi_results = {}
for col in feature_cols[:10]:
    try:
        psi = PSI(X_train[[col]], X_test[[col]])
        psi_results[col] = psi
    except:
        psi_results[col] = np.nan

psi_df = pd.Series(psi_results).sort_values(ascending=False)
print("特征PSI排名（前10）:")
print(psi_df)

特征PSI排名（前10）:
apply_for_admission_score        0.0071
bj_qy24                          0.0038
charging_fail_count_1m           0.0038
charging_fail_count_12m          0.0032
xiaoniu_c3                       0.0030
mobtech80                        0.0019
bairong_v1                       0.0018
apply_for_admission_confidence   0.0001
dxm_v6pro                        0.0000
bcf_fpv1                         0.0000
dtype: float64


## 5. 特征重要性指标

IV等信息价值指标。

In [8]:
# 单特征IV
iv_score = IV(df[target_col], df[demo_feature])
print(f"{demo_feature} IV值: {iv_score:.4f}")

# IV分箱表
iv_table = IV_table(df[[demo_feature]], df[target_col])
print(f"\n{demo_feature} IV分箱表:")
print(iv_table)

bj_qy24 IV值: 0.0901

bj_qy24 IV分箱表:
Empty DataFrame
Columns: []
Index: []


In [9]:
# 批量计算特征IV
iv_results = {}
for col in feature_cols[:10]:
    try:
        iv = IV(df[[col]], df[target_col])
        iv_results[col] = iv
    except:
        iv_results[col] = np.nan

iv_df = pd.Series(iv_results).sort_values(ascending=False)
print("特征IV排名（前10）:")
print(iv_df)

特征IV排名（前10）:
bj_qy24                          NaN
mobtech80                        NaN
bairong_v1                       NaN
xiaoniu_c3                       NaN
dxm_v6pro                        NaN
bcf_fpv1                         NaN
apply_for_admission_confidence   NaN
apply_for_admission_score        NaN
charging_fail_count_12m          NaN
charging_fail_count_1m           NaN
dtype: float64


## 6. 分箱指标

WOE、IV、KS按分箱计算。

In [10]:
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics.binning_metrics import compute_bin_stats, ks_by_bin, chi2_by_bin

# 分箱
binner = OptimalBinning(method='quantile', max_n_bins=5)
binner.fit(df[[demo_feature]], df[target_col])
X_binned = binner.transform(df[demo_feature], metric='indices').values.flatten()

# 计算分箱统计
bin_stats = compute_bin_stats(X_binned, df[target_col])
print(f"{demo_feature} 分箱统计:")
print(bin_stats)

# 分箱KS
ks_bins = ks_by_bin(X_binned, df[target_col])
print(f"\n分箱KS值:")
print(ks_bins)

bj_qy24 分箱统计:
   分箱  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0  2480  2258   222 0.1992 0.1943 0.2688 0.0895  0.3245 0.0242 0.0763 1.3490  0.0868   1.3490  0.0868    2258     222 0.0745
1   1  2489  2296   193 0.2000 0.1976 0.2337 0.0775  0.1678 0.0061 0.0763 1.1686  0.0421   1.2586  0.1718    4554     415 0.1106
2   2  2484  2314   170 0.1996 0.1991 0.2058 0.0684  0.0331 0.0002 0.0763 1.0314  0.0078   1.1829  0.2729    6868     585 0.1173
3   3  2503  2367   136 0.2011 0.2037 0.1646 0.0543 -0.2127 0.0083 0.0763 0.8188 -0.0456   1.0914  0.3650    9235     721 0.0783
4   4  2492  2387   105 0.2002 0.2054 0.1271 0.0421 -0.4798 0.0376 0.0763 0.6350 -0.0914   1.0000  0.0000   11622     826 0.0000

分箱KS值:
(0.11728424383412439, array([0.07447844, 0.11057825, 0.11728424, 0.07826769, 0.        ]))


## 7. 回归指标

MSE、MAE、RMSE、R2等回归评估指标。

In [11]:
# 使用模型预测值作为示例
# 计算回归指标（这里用预测概率作为示例）
y_true_reg = y_test.astype(float)
y_pred_reg = y_pred_proba

mse_score = MSE(y_true_reg, y_pred_reg)
mae_score = MAE(y_true_reg, y_pred_reg)
rmse_score = RMSE(y_true_reg, y_pred_reg)
r2_score = R2(y_true_reg, y_pred_reg)

print("回归评估指标:")
print(f"  MSE: {mse_score:.4f}")
print(f"  MAE: {mae_score:.4f}")
print(f"  RMSE: {rmse_score:.4f}")
print(f"  R2: {r2_score:.4f}")

回归评估指标:
  MSE: 0.0592
  MAE: 0.0618
  RMSE: 0.2433
  R2: 0.0415


## 8. 指标汇总报告

将所有指标汇总保存。

In [12]:
# 指标汇总
metrics_summary = pd.DataFrame({
    '指标': ['KS', 'AUC', 'Gini', 'PSI', 'MSE', 'MAE', 'RMSE'],
    '值': [ks_score, auc_score, gini_score, psi_score, mse_score, mae_score, rmse_score]
})

print("指标汇总:")
print(metrics_summary.to_string(index=False))

# 保存结果 - 使用hscredit的ExcelWriter和DataFrame.save()
from hscredit.report.excel import ExcelWriter

output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/metrics_results.xlsx'

# 使用with语句（推荐，自动保存）
with ExcelWriter(theme_color='2639E9').set_filename(output_path) as writer:
    # 使用DataFrame.save()方法保存
    metrics_summary.save(writer, sheet_name='指标汇总', title='指标汇总', auto_width=True, condition_cols=['值'])
    psi_df.to_frame('PSI值').save(writer, sheet_name='特征PSI', title='特征PSI排名', auto_width=True)
    iv_df.to_frame('IV值').save(writer, sheet_name='特征IV', title='特征IV排名', auto_width=True, condition_cols=['IV值'])
    bin_stats.save(writer, sheet_name='分箱统计', title='分箱统计', auto_width=True)

print(f"\n指标结果已保存至: {output_path}")

指标汇总:
  指标       值
  KS  0.1641
 AUC  0.4936
Gini -0.0127
 PSI  0.0000
 MSE  0.0592
 MAE  0.0618
RMSE  0.2433

指标结果已保存至: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/metrics_results.xlsx
